# `vad_mlx_hybrid.ipynb` 使用說明

這個 notebook 用來處理中藏雙語音檔，但改成混合架構：`whisperx` 負責 VAD / LID / alignment，`mlx_whisper` 負責 Apple Silicon / Metal 上的 ASR。

## Notebook 流程
1. 指定原始 MP3 檔名並切成多個片段。
2. 針對每個 10 分鐘切片依序做 VAD，找出實際有人聲的時間區段。
3. 用 `whisperx` 的 `zh` 輔助模型做 LID，把每段標成 `zh` 或 `bo`。
4. 依語言把片段分流到對應的 MLX Whisper 模型做轉寫。
5. 把 MLX 產生的 segments 交給 `whisperx.align` 做時間對齊，最後整合回原始音檔的全域時間軸。

> 這個版本假設 `bo` 模型已先轉成 MLX 格式；若還沒有，請先用 MLX Whisper 的 `convert.py` 把 Hugging Face / PyTorch 權重轉成 MLX 權重。


## 1. 準備輸入音檔與切段

- `mp3_audio` 是原始輸入檔名。
- `trim_duration` 目前設成 `60 * 10`，也就是每 10 分鐘切一段。
- 產出的切片會寫到 `audio/`，檔名格式為 `{index}_{start}_{end}.mp3`。

這一段的目的是先把長音檔切成較容易處理的片段，避免後面 VAD 與 ASR 一次吃進過長的音訊。


In [1]:
# Add mp3 audio file name
mp3_audio = "910810a_P077.mp3"

In [2]:
import ffmpeg
import math
import os

In [3]:
trim_duration = 60 * 10
output_dir = "audio"
os.makedirs("audio", exist_ok=True)
probe = ffmpeg.probe(mp3_audio)
stream = next((stream for stream in probe['streams'] if stream['codec_type'] == 'audio'), None)
duration = math.floor(float(stream['duration']))
print(f"Total duration: {duration}")
print(f"Trim into {math.ceil(duration/trim_duration)} files by {trim_duration} mins")

audio_input = ffmpeg.input(mp3_audio)
flag = 0
for cut in range(math.ceil(duration/trim_duration)):
    if flag+trim_duration > duration:
        end = duration
    else:
        end = flag + trim_duration
    audio_cut = audio_input.audio.filter('atrim', start=flag, end=end)
    audio_output = ffmpeg.output(audio_cut, f'audio/{cut}_{flag}_{end}.mp3')
    ffmpeg.run(audio_output)
    flag += trim_duration

Total duration: 3780
Trim into 7 files by 600 mins


ffmpeg version 5.1.6 Copyright (c) 2000-2024 the FFmpeg developers
  built with Apple clang version 16.0.0 (clang-1600.0.26.6)
  configuration: --prefix='/usr/local/Cellar/ffmpeg@5/5.1.6_7' --datadir='/usr/local/Cellar/ffmpeg@5/5.1.6_7/share/ffmpeg' --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags='-Wl,-ld_classic' --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-

Error: ffmpeg error (see stderr output for detail)

In [4]:
import glob
audio_cuts = glob.glob("audio/*.mp3")
audio_cuts = sorted(
    audio_cuts,
    key=lambda x: tuple(int(v) for v in os.path.splitext(os.path.basename(x))[0].split("_")[:3]),
)
print(audio_cuts)
print(f"find totally audio_cut: {len(audio_cuts)}")

['audio/0_0_600.mp3', 'audio/1_600_1200.mp3', 'audio/2_1200_1800.mp3', 'audio/3_1800_2400.mp3', 'audio/4_2400_3000.mp3', 'audio/5_3000_3600.mp3', 'audio/6_3600_3780.mp3']
find totally audio_cut: 7


## 2. 核心工具函式

這個區塊把整個流程需要的 helper function 先定義好，後面的 cell 只要專注在串流程。

重點函式如下：
- `run_vad_segments`：用 VAD 找出有人聲的區段。
- `detect_zh_bo`：用滑動視窗累積語言分數，做 `zh` / `bo` 二分類。
- `build_lid_blocks`、`smooth_langs_binary`：把零碎語音段整理成較穩定的語言標記。
- `merge_for_asr`：把相鄰且同語言的區段合併，降低 ASR 呼叫次數。
- `transcribe_block_fast`、`transcribe_and_align`：先用 MLX 做 ASR，再交給 `whisperx.align` 對齊。
- `parse_audio_cut_path`、`shift_time_rows`、`run_pipeline_for_audio_cut`：把每個切片結果換算回全域時間並整合。
- `write_csv`、`write_srt`：把結果輸出成後處理容易使用的格式。


In [6]:
def normalize_lang(lang: str) -> str:
    lang = (lang or "").lower()
    if lang.startswith("zh"):
        return "zh"
    if lang.startswith("bo"):
        return "bo"
    return "other"


def detect_language_no_warning(pipeline, audio_chunk):
    model_n_mels = pipeline.model.feat_kwargs.get("feature_size")
    segment = log_mel_spectrogram(
        audio_chunk[:N_SAMPLES],
        n_mels=model_n_mels if model_n_mels is not None else 80,
        padding=0 if audio_chunk.shape[0] >= N_SAMPLES else N_SAMPLES - audio_chunk.shape[0],
    )
    encoder_output = pipeline.model.encode(segment)
    results = pipeline.model.model.detect_language(encoder_output)
    language_token, probability = results[0][0]
    return language_token[2:-2], float(probability)


def run_vad_segments(audio, vad_model, chunk_size=30, vad_onset=0.5, vad_offset=0.363):
    if issubclass(type(vad_model), Vad):
        waveform = vad_model.preprocess_audio(audio)
        merge_chunks = vad_model.merge_chunks
    else:
        waveform = Pyannote.preprocess_audio(audio)
        merge_chunks = Pyannote.merge_chunks

    vad_scores = vad_model({"waveform": waveform, "sample_rate": SAMPLE_RATE})
    merged = merge_chunks(vad_scores, chunk_size, onset=vad_onset, offset=vad_offset)

    segs = []
    for s in merged:
        start = float(s["start"])
        end = float(s["end"])
        if end - start >= 0.2:
            segs.append({"start": start, "end": end})
    return segs


def _lang_probs(pipeline, chunk):
    n_mels = pipeline.model.feat_kwargs.get("feature_size") or 80
    mel = log_mel_spectrogram(
        chunk[:N_SAMPLES],
        n_mels=n_mels,
        padding=max(0, N_SAMPLES - len(chunk)),
    )
    enc = pipeline.model.encode(mel)
    return [(tok[2:-2], float(p)) for tok, p in pipeline.model.model.detect_language(enc)[0]]


def detect_zh_bo(
    pipeline,
    chunk,
    win_sec=5.0,
    hop_sec=2.5,
    min_win_sec=0.5,
    zh_low_th=0.85,
    other_min_th=0.15,
):
    votes = {"zh": 0.0, "bo": 0.0, "other": 0.0}
    win = int(win_sec * SAMPLE_RATE)
    hop = int(hop_sec * SAMPLE_RATE)
    min_len = int(min_win_sec * SAMPLE_RATE)

    if len(chunk) <= win:
        ranges = [(0, len(chunk))]
    else:
        ranges = [(i, min(i + win, len(chunk))) for i in range(0, len(chunk) - win + 1, hop)]

    for a, b in ranges:
        sub = chunk[a:b]
        if len(sub) < min_len:
            continue
        for lang, prob in _lang_probs(pipeline, sub):
            if lang.startswith("zh"):
                votes["zh"] += prob
            elif lang.startswith("bo"):
                votes["bo"] += prob
            else:
                votes["other"] += prob

    total = votes["zh"] + votes["bo"] + votes["other"]
    if total == 0:
        return "other", 0.0, votes

    zh_r = votes["zh"] / total
    bo_r = votes["bo"] / total
    other_r = votes["other"] / total

    if zh_r < zh_low_th and other_r >= other_min_th:
        return "bo", 1.0 - zh_r, votes

    if bo_r >= zh_r:
        return "bo", bo_r, votes
    return "zh", zh_r, votes


def build_lid_blocks(vad_segments, min_lid_sec=2.0, max_gap=0.4, max_block_sec=25.0):
    blocks = []
    i, n = 0, len(vad_segments)
    while i < n:
        st = vad_segments[i]["start"]
        ed = vad_segments[i]["end"]
        idxs = [i]
        j = i + 1
        while j < n:
            gap = vad_segments[j]["start"] - ed
            new_ed = vad_segments[j]["end"]
            if gap <= max_gap and (new_ed - st) <= max_block_sec and (ed - st) < min_lid_sec:
                ed = new_ed
                idxs.append(j)
                j += 1
            else:
                break
        blocks.append({"start": st, "end": ed, "idxs": idxs})
        i = j
    return blocks


def detect_block_language_zh_bo(lid_pipeline, audio, start, end):
    s1, s2 = int(start * SAMPLE_RATE), int(end * SAMPLE_RATE)
    chunk = audio[s1:s2]
    if len(chunk) < int(0.4 * SAMPLE_RATE):
        return "other", 0.0, {"zh": 0.0, "bo": 0.0}
    return detect_zh_bo(lid_pipeline, chunk)


def smooth_langs_binary(seg_lang, default_lang="zh"):
    out = seg_lang[:]
    n = len(out)

    for i in range(n):
        if out[i] in {"zh", "bo"}:
            continue
        left = next((out[j] for j in range(i - 1, -1, -1) if out[j] in {"zh", "bo"}), None)
        right = next((out[j] for j in range(i + 1, n) if out[j] in {"zh", "bo"}), None)
        if left == right and left is not None:
            out[i] = left

    for i in range(n):
        if out[i] not in {"zh", "bo"}:
            out[i] = out[i - 1] if i > 0 else default_lang

    return out


def merge_for_asr(vad_segments, seg_lang, merge_gap=0.5, min_asr_sec=0.3, max_block_sec=45.0):
    rows = []
    for s, lang in zip(vad_segments, seg_lang):
        if lang not in {"zh", "bo"}:
            continue
        rows.append({"start": s["start"], "end": s["end"], "lang": lang})

    if not rows:
        return []

    merged = [rows[0]]
    for cur in rows[1:]:
        prev = merged[-1]
        same_lang = cur["lang"] == prev["lang"]
        gap = cur["start"] - prev["end"]
        new_len = cur["end"] - prev["start"]
        if same_lang and gap <= merge_gap and new_len <= max_block_sec:
            prev["end"] = cur["end"]
        else:
            merged.append(cur)

    return [m for m in merged if (m["end"] - m["start"]) >= min_asr_sec]


def transcribe_block_fast(model_ref, audio_chunk, lang):
    result = mlx_whisper.transcribe(
        audio_chunk,
        path_or_hf_repo=model_ref,
        language=lang,
        task="transcribe",
        word_timestamps=False,
        condition_on_previous_text=False,
        verbose=False,
    )

    segments = []
    for s in result.get("segments", []):
        text = (s.get("text") or "").strip()
        if text:
            segments.append({
                "start": float(s["start"]),
                "end": float(s["end"]),
                "text": text,
            })
    return segments


def transcribe_and_align(audio, asr_blocks, asr_models, align_models, device):
    rows = []
    for block in asr_blocks:
        lang = block["lang"]
        s1 = int(block["start"] * SAMPLE_RATE)
        s2 = int(block["end"] * SAMPLE_RATE)
        audio_chunk = audio[s1:s2]
        if len(audio_chunk) < int(0.4 * SAMPLE_RATE):
            continue

        base_segments = transcribe_block_fast(asr_models[lang], audio_chunk, lang)
        if not base_segments:
            continue

        aligned_segments = base_segments
        if align_models.get(lang) is not None:
            model_a, metadata = align_models[lang]
            try:
                aligned = whisperx.align(
                    base_segments,
                    model_a,
                    metadata,
                    audio_chunk,
                    device,
                    return_char_alignments=False,
                )
                aligned_segments = aligned.get("segments", base_segments)
            except Exception:
                aligned_segments = base_segments

        for s in aligned_segments:
            text = (s.get("text") or "").strip()
            if not text:
                continue
            st = round(block["start"] + float(s["start"]), 3)
            en = round(block["start"] + float(s["end"]), 3)
            if en <= st:
                continue
            rows.append({"start": st, "end": en, "lang": lang, "text": text})

    rows.sort(key=lambda x: (x["start"], x["end"]))
    return rows


def parse_audio_cut_path(audio_path):
    stem = os.path.splitext(os.path.basename(audio_path))[0]
    parts = stem.split("_")
    if len(parts) < 3:
        raise ValueError(f"Unexpected audio cut name: {audio_path}")
    chunk_index, chunk_start, chunk_end = parts[:3]
    return {
        "audio_path": audio_path,
        "audio_file": os.path.basename(audio_path),
        "chunk_index": int(chunk_index),
        "chunk_start": float(chunk_start),
        "chunk_end": float(chunk_end),
    }


def shift_time_rows(rows, offset_sec, extra_fields=None):
    shifted = []
    for row in rows:
        item = row.copy()
        if "start" in item:
            item["start"] = round(float(item["start"]) + offset_sec, 3)
        if "end" in item:
            item["end"] = round(float(item["end"]) + offset_sec, 3)
        if extra_fields:
            item.update(extra_fields)
        shifted.append(item)
    return shifted


def run_pipeline_for_audio_cut(audio_path, zh_model, asr_models, align_models, device):
    chunk_info = parse_audio_cut_path(audio_path)
    audio = whisperx.load_audio(audio_path)

    vad_segments = run_vad_segments(
        audio,
        zh_model.vad_model,
        chunk_size=30,
        vad_onset=0.5,
        vad_offset=0.363,
    )

    lid_pipeline = zh_model
    lid_blocks = build_lid_blocks(vad_segments, min_lid_sec=2.0, max_gap=0.3, max_block_sec=25.0)

    seg_lang = ["other"] * len(vad_segments)
    seg_conf = [0.0] * len(vad_segments)

    for block in lid_blocks:
        lang, conf, _ = detect_block_language_zh_bo(
            lid_pipeline, audio, block["start"], block["end"]
        )
        for idx in block["idxs"]:
            seg_lang[idx] = lang
            seg_conf[idx] = conf

    seg_lang = smooth_langs_binary(seg_lang, default_lang="zh")

    timeline_rows = []
    for i, (seg, lang) in enumerate(zip(vad_segments, seg_lang)):
        timeline_rows.append({
            "start": round(seg["start"], 3),
            "end": round(seg["end"], 3),
            "duration": round(seg["end"] - seg["start"], 3),
            "lang": lang,
            "lid_conf": round(seg_conf[i], 3),
        })

    local_asr_blocks = merge_for_asr(
        vad_segments,
        seg_lang,
        merge_gap=0.5,
        min_asr_sec=1.0,
        max_block_sec=45.0,
    )
    transcript_rows = transcribe_and_align(audio, local_asr_blocks, asr_models, align_models, device)

    extra_fields = {
        "chunk_index": chunk_info["chunk_index"],
        "audio_file": chunk_info["audio_file"],
    }
    timeline_rows = shift_time_rows(timeline_rows, chunk_info["chunk_start"], extra_fields)
    asr_blocks = shift_time_rows(local_asr_blocks, chunk_info["chunk_start"], extra_fields)
    transcript_rows = shift_time_rows(transcript_rows, chunk_info["chunk_start"], extra_fields)

    summary = {
        "chunk_index": chunk_info["chunk_index"],
        "audio_file": chunk_info["audio_file"],
        "chunk_start": chunk_info["chunk_start"],
        "chunk_end": chunk_info["chunk_end"],
        "vad_segments": len(vad_segments),
        "asr_blocks": len(asr_blocks),
        "transcript_rows": len(transcript_rows),
        "zh_segments": sum(1 for row in timeline_rows if row["lang"] == "zh"),
        "bo_segments": sum(1 for row in timeline_rows if row["lang"] == "bo"),
    }

    result = dict(chunk_info)
    result["summary"] = summary
    result["timeline_rows"] = timeline_rows
    result["asr_blocks"] = asr_blocks
    result["transcript_rows"] = transcript_rows
    return result


def sec_to_srt_time(sec):
    sec = max(0.0, float(sec))
    whole = int(sec)
    ms = int(round((sec - whole) * 1000))
    if ms == 1000:
        whole += 1
        ms = 0
    s = whole % 60
    m = (whole // 60) % 60
    h = whole // 3600
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"


def merge_subtitle_rows(rows, max_gap=0.35, max_block_sec=10.0):
    if not rows:
        return []
    out = [rows[0].copy()]
    for cur in rows[1:]:
        prev = out[-1]
        same_lang = cur["lang"] == prev["lang"]
        gap = cur["start"] - prev["end"]
        new_len = cur["end"] - prev["start"]
        if same_lang and gap <= max_gap and new_len <= max_block_sec:
            prev["end"] = cur["end"]
            if cur["lang"] in {"zh", "bo"}:
                prev["text"] = (prev["text"] + cur["text"]).strip()
            else:
                prev["text"] = (prev["text"] + " " + cur["text"]).strip()
        else:
            out.append(cur.copy())
    return out


def write_csv(rows, path, fieldnames):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)


def write_srt(rows, path):
    with open(path, "w", encoding="utf-8") as f:
        for i, r in enumerate(rows, 1):
            lang_tag = "漢語" if r["lang"] == "zh" else "藏語" if r["lang"] == "bo" else r["lang"]
            f.write(f"{i}\n")
            f.write(f"{sec_to_srt_time(r['start'])} --> {sec_to_srt_time(r['end'])}\n")
            f.write(f"[{lang_tag}] {r['text'].strip()}\n\n")


## 3. 模型與執行參數

這裡集中設定本次執行會用到的音檔、輸出資料夾，以及 `whisperx` 與 MLX 兩套模型。

常用調整項目：
- `AUDIO_PATHS`：要批次處理的所有切片，預設就是 `audio_cuts`。
- `ZH_HELPER_MODEL`：`whisperx` 用來做 VAD / LID 的中文輔助模型。
- `ZH_MLX_ASR_MODEL`、`BO_MLX_ASR_MODEL`：中文與藏文的 MLX Whisper 轉寫模型。
- `ZH_ALIGN_MODEL`、`BO_ALIGN_MODEL`：需要更準時間戳時使用的 `whisperx` 對齊模型。
- `DEVICE`、`COMPUTE_TYPE`：`whisperx` 的執行裝置與量化設定；MLX 會自動走 Apple Silicon。


In [8]:
# 如果尚未安裝：
# !pip install mlx mlx-whisper whisperx pyannote.audio

import os
from pathlib import Path
import warnings
import logging
import csv
import json

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("XDG_CACHE_HOME", str((Path.cwd() / ".cache").resolve()))
os.environ.setdefault("MPLCONFIGDIR", str((Path.cwd() / ".mplconfig").resolve()))
Path(os.environ["XDG_CACHE_HOME"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

warnings.filterwarnings(
    "ignore",
    message=r"\s*torchcodec is not installed correctly so built-in audio decoding will fail\.",
    module=r"pyannote\.audio\.core\.io",
)
warnings.filterwarnings(
    "ignore",
    message=r"torchaudio\._backend\.list_audio_backends has been deprecated\..*",
    module=r"speechbrain\.utils\.torch_audio_backend",
)
logging.getLogger("speechbrain.utils.quirks").setLevel(logging.ERROR)

import whisperx
import mlx_whisper
from whisperx.audio import SAMPLE_RATE, N_SAMPLES, log_mel_spectrogram
from whisperx.vads import Vad, Pyannote

AUDIO_PATHS = audio_cuts
OUT_DIR = "./out_whisper_mlx"

DEVICE = "cpu"                  # whisperx 的 VAD / LID / align 在 Mac 先走 cpu 較穩
COMPUTE_TYPE = "int8"

ZH_HELPER_MODEL = "deepdml/faster-whisper-large-v3-turbo-ct2"

ZH_MLX_ASR_MODEL = "mlx-community/whisper-large-v3-turbo"
BO_MLX_ASR_MODEL = "./mlx_models/openpecha_op-whisper_small-ft-v2"  # 先轉成 MLX 格式後填入

ZH_ALIGN_MODEL = "jonatasgrosman/wav2vec2-large-xlsr-53-chinese-zh-cn"
BO_ALIGN_MODEL = "openpecha/op-wav2vec2-ft-v17"

if not AUDIO_PATHS:
    raise ValueError("No audio cuts found. Run the slicing cells first.")

os.makedirs(OUT_DIR, exist_ok=True)


In [10]:
zh_model = whisperx.load_model(
    ZH_HELPER_MODEL,
    DEVICE,
    compute_type=COMPUTE_TYPE,
    language="zh",
)

asr_models = {"zh": ZH_MLX_ASR_MODEL, "bo": BO_MLX_ASR_MODEL}

align_models = {}
for lang, model_name in [("zh", ZH_ALIGN_MODEL), ("bo", BO_ALIGN_MODEL)]:
    try:
        align_models[lang] = whisperx.load_align_model(
            language_code=lang,
            device=DEVICE,
            model_name=model_name,
        )
        print(f"{lang} align model loaded")
    except Exception as e:
        align_models[lang] = None
        print(f"[WARN] {lang} align unavailable:", e)

print(f"audio cuts to process: {len(AUDIO_PATHS)}")
print("first audio cut:", AUDIO_PATHS[0])
print("MLX ASR models:", asr_models)


tokenizer.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

vocabulary.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.bin:   0%|          | 0.00/1.62G [00:00<?, ?B/s]

2026-03-15 00:29:12 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


... (13 outputs left out)

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

bo align model loaded
audio cuts to process: 7
first audio cut: audio/0_0_600.mp3
MLX ASR models: {'zh': 'mlx-community/whisper-large-v3-turbo', 'bo': './mlx_models/openpecha_op-whisper_small-ft-v2'}


## 4. 載入模型並批次執行流程

這個區塊會先載入 `whisperx` 的 VAD / LID / alignment 模型，再用 `for` 迴圈依序處理 `AUDIO_PATHS` 裡的每個 10 分鐘切片。

每個切片都會經過 VAD、LID、MLX ASR 與 `whisperx.align`，最後再把時間戳加回原始音檔的全域時間。`run_vad_segments` 仍然會忽略短於 0.2 秒的片段，避免太短的噪音影響結果。


In [11]:
chunk_results = []

for audio_path in [AUDIO_PATHS[0]]:
    result = run_pipeline_for_audio_cut(
        audio_path,
        zh_model=zh_model,
        asr_models=asr_models,
        align_models=align_models,
        device=DEVICE,
    )
    chunk_results.append(result)

    summary = result["summary"]
    print(
        f"[{summary['chunk_index']}] {summary['audio_file']} | "
        f"VAD={summary['vad_segments']} | "
        f"ASR={summary['asr_blocks']} | "
        f"Transcript={summary['transcript_rows']} | "
        f"ZH={summary['zh_segments']} | "
        f"BO={summary['bo_segments']}"
    )

chunk_summaries = [result["summary"] for result in chunk_results]
print("chunks processed:", len(chunk_results))
chunk_summaries[:3]


100%|██████████| 2219/2219 [00:09<00:00, 231.66frames/s]


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

.gitattributes: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/268 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/316 [00:00<?, ?B/s]

weights.safetensors:   0%|          | 0.00/1.61G [00:00<?, ?B/s]

100%|██████████| 2809/2809 [00:09<00:00, 291.26frames/s]


2026-03-15 00:41:16 - whisperx.alignment - WARNING - Failed to align segment ("red zer gi yod red zer gi yod red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red"): backtrack failed, resorting to origi

100%|██████████| 1655/1655 [00:09<00:00, 173.43frames/s]


[0] 0_0_600.mp3 | VAD=25 | ASR=20 | Transcript=144 | ZH=12 | BO=13
chunks processed: 1


[{'chunk_index': 0,
  'audio_file': '0_0_600.mp3',
  'chunk_start': 0.0,
  'chunk_end': 600.0,
  'vad_segments': 25,
  'asr_blocks': 20,
  'transcript_rows': 144,
  'zh_segments': 12,
  'bo_segments': 13}]

## 5. 檢查語言時間軸（LID）

上一個 cell 已經把每個切片的 VAD / LID / MLX ASR / alignment 都跑完，這裡開始把 LID 結果整理成同一條全域時間軸。

`timeline_rows` 會保留 `chunk_index` 與 `audio_file`，方便回頭定位是哪一個 10 分鐘切片；而 `start` / `end` 則已經換算成原始長音檔的時間。


In [12]:
timeline_rows = []
for result in chunk_results:
    timeline_rows.extend(result["timeline_rows"])

timeline_rows.sort(key=lambda x: (x["start"], x["end"]))
print("timeline rows:", len(timeline_rows))
print(chunk_summaries[:3])
timeline_rows[:10]


timeline rows: 25
[{'chunk_index': 0, 'audio_file': '0_0_600.mp3', 'chunk_start': 0.0, 'chunk_end': 600.0, 'vad_segments': 25, 'asr_blocks': 20, 'transcript_rows': 144, 'zh_segments': 12, 'bo_segments': 13}]


[{'start': 0.031,
  'end': 28.583,
  'duration': 28.553,
  'lang': 'bo',
  'lid_conf': 0.626,
  'chunk_index': 0,
  'audio_file': '0_0_600.mp3'},
 {'start': 29.123,
  'end': 54.638,
  'duration': 25.515,
  'lang': 'bo',
  'lid_conf': 0.827,
  'chunk_index': 0,
  'audio_file': '0_0_600.mp3'},
 {'start': 54.773,
  'end': 80.339,
  'duration': 25.566,
  'lang': 'bo',
  'lid_conf': 0.798,
  'chunk_index': 0,
  'audio_file': '0_0_600.mp3'},
 {'start': 80.744,
  'end': 108.841,
  'duration': 28.097,
  'lang': 'bo',
  'lid_conf': 0.78,
  'chunk_index': 0,
  'audio_file': '0_0_600.mp3'},
 {'start': 109.752,
  'end': 138.98,
  'duration': 29.228,
  'lang': 'bo',
  'lid_conf': 0.758,
  'chunk_index': 0,
  'audio_file': '0_0_600.mp3'},
 {'start': 140.144,
  'end': 168.005,
  'duration': 27.861,
  'lang': 'bo',
  'lid_conf': 0.752,
  'chunk_index': 0,
  'audio_file': '0_0_600.mp3'},
 {'start': 168.393,
  'end': 189.74,
  'duration': 21.347,
  'lang': 'bo',
  'lid_conf': 0.852,
  'chunk_index': 0,


In [13]:
print(timeline_rows[:10])

# 1) 語言時間軸 CSV
timeline_csv = os.path.join(OUT_DIR, "speech_lang_timeline.csv")
write_csv(
    timeline_rows,
    timeline_csv,
    ["chunk_index", "audio_file", "start", "end", "duration", "lid_conf", "lang"],
)
timeline_csv


[{'start': 0.031, 'end': 28.583, 'duration': 28.553, 'lang': 'bo', 'lid_conf': 0.626, 'chunk_index': 0, 'audio_file': '0_0_600.mp3'}, {'start': 29.123, 'end': 54.638, 'duration': 25.515, 'lang': 'bo', 'lid_conf': 0.827, 'chunk_index': 0, 'audio_file': '0_0_600.mp3'}, {'start': 54.773, 'end': 80.339, 'duration': 25.566, 'lang': 'bo', 'lid_conf': 0.798, 'chunk_index': 0, 'audio_file': '0_0_600.mp3'}, {'start': 80.744, 'end': 108.841, 'duration': 28.097, 'lang': 'bo', 'lid_conf': 0.78, 'chunk_index': 0, 'audio_file': '0_0_600.mp3'}, {'start': 109.752, 'end': 138.98, 'duration': 29.228, 'lang': 'bo', 'lid_conf': 0.758, 'chunk_index': 0, 'audio_file': '0_0_600.mp3'}, {'start': 140.144, 'end': 168.005, 'duration': 27.861, 'lang': 'bo', 'lid_conf': 0.752, 'chunk_index': 0, 'audio_file': '0_0_600.mp3'}, {'start': 168.393, 'end': 189.74, 'duration': 21.347, 'lang': 'bo', 'lid_conf': 0.852, 'chunk_index': 0, 'audio_file': '0_0_600.mp3'}, {'start': 190.381, 'end': 212.572, 'duration': 22.191, 'la

'./out/speech_lang_timeline.csv'

## 6. 整理轉寫、對齊與輸出結果

這幾個 cell 會把每個切片的批次處理結果攤平成單一清單，輸出成全域時間的 JSON、CSV 與 SRT。

輸出內容包含：
- `asr_blocks.json`：實際拿去轉寫的區段。
- `bilingual_transcript.csv`：逐段轉寫結果，適合人工校對。
- `bilingual_transcript.json`：保留結構化欄位，方便後續程式處理。
- `bilingual_transcript.srt`：可直接拿去播放器或剪輯軟體使用的字幕檔。

所有 `start` / `end` 都已經換算回原始長音檔的時間軸；ASR 來自 MLX，而對齊仍由 `whisperx.align` 負責。


In [14]:
asr_blocks = []
for result in chunk_results:
    asr_blocks.extend(result["asr_blocks"])

asr_blocks.sort(key=lambda x: (x["start"], x["end"]))
print("ASR blocks:", len(asr_blocks))
asr_blocks[:5]


ASR blocks: 20


[{'start': 0.031,
  'end': 28.583,
  'lang': 'bo',
  'chunk_index': 0,
  'audio_file': '0_0_600.mp3'},
 {'start': 29.123,
  'end': 54.638,
  'lang': 'bo',
  'chunk_index': 0,
  'audio_file': '0_0_600.mp3'},
 {'start': 54.773,
  'end': 80.339,
  'lang': 'bo',
  'chunk_index': 0,
  'audio_file': '0_0_600.mp3'},
 {'start': 80.744,
  'end': 108.841,
  'lang': 'bo',
  'chunk_index': 0,
  'audio_file': '0_0_600.mp3'},
 {'start': 109.752,
  'end': 138.98,
  'lang': 'bo',
  'chunk_index': 0,
  'audio_file': '0_0_600.mp3'}]

In [15]:
asr_blocks_json_path = os.path.join(OUT_DIR, "asr_blocks.json")
with open(asr_blocks_json_path, "w", encoding="utf-8") as f:
    json.dump(asr_blocks, f, ensure_ascii=False, indent=2)

asr_blocks_json_path


'./out/asr_blocks.json'

In [16]:
transcript_rows = []
for result in chunk_results:
    transcript_rows.extend(result["transcript_rows"])

transcript_rows.sort(key=lambda x: (x["start"], x["end"]))
print("transcript rows:", len(transcript_rows))
print(transcript_rows[:10])

# 2) 逐字稿 CSV + JSON
transcript_csv = os.path.join(OUT_DIR, "bilingual_transcript.csv")
write_csv(
    transcript_rows,
    transcript_csv,
    ["chunk_index", "audio_file", "start", "end", "lang", "text"],
)

transcript_json = os.path.join(OUT_DIR, "bilingual_transcript.json")
with open(transcript_json, "w", encoding="utf-8") as f:
    json.dump(transcript_rows, f, ensure_ascii=False, indent=2)


transcript rows: 144
[{'start': 0.031, 'end': 28.601, 'lang': 'bo', 'text': "a le sngan la ya ga re zer dgos red chos nyan shul ma byas gong la ya nga rang shul gnyal yin na'ang red rnam pa tsho kun spyod gnyis ki nang nas kun spyod gnyis ki nang nas kun spyod gnyis ki nang nas kun spyod gnyis ki nang nas kun spyod gnyis ki nang nas kun spyod gnyis ki nang nas kun spyod gnyis ki nang nas kun spyod gnyis ki nang nas kun spyod gnyis ki nang nas kun spyod gnyis ki nang nas yong gi red zer red zer gong red zer gong red zer red zer gong red zer red zer gong red zer red zer gong red zer red zer red zer gong red zer red zer red zer gong red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red zer red", 'chunk_index': 0, 'audio_file': '0_0_600.mp3'}, {'start': 29.123, 'end': 54.653, 'lang': 'bo', 'text': "gcig ma gtogs ga yod red nga mar snga ma nas bod ki ga re zer dgos red da sangs rgyas ki bstan pa dar nas bzung ba cig gi dus da lta

In [17]:
# 3) 合併後 SRT
srt_rows = merge_subtitle_rows(transcript_rows, max_gap=0.35, max_block_sec=10.0)
srt_path = os.path.join(OUT_DIR, "bilingual_transcript.srt")
write_srt(srt_rows, srt_path)

print("Done")
print(" -", timeline_csv)
print(" -", asr_blocks_json_path)
print(" -", transcript_csv)
print(" -", transcript_json)
print(" -", srt_path)


Done
 - ./out/speech_lang_timeline.csv
 - ./out/asr_blocks.json
 - ./out/bilingual_transcript.csv
 - ./out/bilingual_transcript.json
 - ./out/bilingual_transcript.srt
